# Feature Exploration
Visualizing all 24 engineered features for AAPL.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings("ignore")

df = pd.read_parquet("data/processed/AAPL_features.parquet")
print(f"Shape: {df.shape}")
print(f"Features: {[c for c in df.columns if c not in ["open","high","low","close","volume","ticker"]]}")
print("
Null rates (%):")
print((df.isnull().mean()*100).round(2))

## Technical Indicators

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=["RSI-14", "MACD Histogram", "Bollinger %B"])
fig.add_trace(go.Scatter(x=df.index, y=df["rsi_14"], line=dict(color="#89b4fa")), row=1, col=1)
fig.add_hline(y=70, line_dash="dot", line_color="red", row=1, col=1)
fig.add_hline(y=30, line_dash="dot", line_color="green", row=1, col=1)
fig.add_trace(go.Bar(x=df.index, y=df["macd_histogram"],
                     marker_color=np.where(df["macd_histogram"]>=0, "#a6e3a1", "#f38ba8")), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["bb_pct_b"], line=dict(color="#cba6f7")), row=3, col=1)
fig.add_hline(y=1, line_dash="dot", row=3, col=1)
fig.add_hline(y=0, line_dash="dot", row=3, col=1)
fig.update_layout(template="plotly_dark", height=600, title="Technical Indicators — AAPL", showlegend=False)
fig.show()

## Volatility Features

In [ ]:
fig = go.Figure()
for col, color in [("realized_vol_10d","#89b4fa"),("realized_vol_30d","#a6e3a1"),("parkinson_vol","#fab387")]:
    fig.add_trace(go.Scatter(x=df.index, y=df[col]*100, name=col, line=dict(color=color)))
fig.update_layout(template="plotly_dark", title="Annualized Volatility Estimates (%)",
                  xaxis_title="Date", yaxis_title="Volatility (%)", height=400)
fig.show()

## Statistical Features: Hurst Exponent & Entropy

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Hurst Exponent (>0.5=trending, <0.5=mean-reverting)",
                                    "Sample Entropy (higher=more complex/random)"])
fig.add_trace(go.Scatter(x=df.index, y=df["hurst_exponent"], line=dict(color="#89b4fa")), row=1, col=1)
fig.add_hline(y=0.5, line_dash="dot", line_color="white", row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["sample_entropy"], line=dict(color="#a6e3a1")), row=2, col=1)
fig.update_layout(template="plotly_dark", height=500, showlegend=False,
                  title="Regime-Detecting Statistical Features — AAPL")
fig.show()

## Feature Correlation Heatmap

In [ ]:
from src.features.constants import FEATURE_COLS
feat_cols = [c for c in FEATURE_COLS if c in df.columns]
corr = df[feat_cols].corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Feature Correlation Matrix (24 features)", template="plotly_dark",
                width=900, height=800)
fig.update_traces(text=corr.round(2), texttemplate="%{text}")
fig.show()